# SadTalker from existing audio on Kaggle

Use this if you already have `audio.wav` and `portrait.jpg` in a private Kaggle dataset. Includes the compatibility patches needed for modern Kaggle/Python 3.12 and H.264 output recoding.

In [ ]:

import json, os, pathlib, shutil, subprocess, sys, traceback
WORK = pathlib.Path('/kaggle/working')
FINAL = WORK / 'result_sadtalker_from_audio_h264.mp4'
MANIFEST = WORK / 'manifest.json'

def sh(cmd, **kw):
    print('\n$', cmd if isinstance(cmd, str) else ' '.join(map(str, cmd)), flush=True)
    return subprocess.check_call(cmd, **kw)

def out(cmd):
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)

def find_asset(name):
    hits = []
    for root in [pathlib.Path('/kaggle/input'), pathlib.Path('/kaggle/working'), pathlib.Path('/kaggle/src')]:
        if root.exists(): hits.extend(root.rglob(name))
    if not hits: raise FileNotFoundError(name)
    print('asset', name, '->', hits[0])
    return hits[0]

def patch_sadtalker():
    import site
    for base in site.getsitepackages():
        ft = pathlib.Path(base) / 'torchvision/transforms/functional_tensor.py'
        if ft.parent.exists() and not ft.exists():
            ft.write_text('from .functional import *\n')
    for pyfile in pathlib.Path('.').rglob('*.py'):
        txt = pyfile.read_text(errors='ignore')
        ntxt = (txt
            .replace('np.VisibleDeprecationWarning', 'DeprecationWarning')
            .replace('np.float,', 'float,').replace('np.float)', 'float)').replace('np.float]', 'float]').replace('np.float ', 'float ')
            .replace('np.int,', 'int,').replace('np.int)', 'int)').replace('np.int]', 'int]').replace('np.int ', 'int ')
            .replace('np.bool,', 'bool,').replace('np.bool)', 'bool)').replace('np.bool]', 'bool]').replace('np.bool ', 'bool '))
        if ntxt != txt: pyfile.write_text(ntxt)
    pre = pathlib.Path('src/face3d/util/preprocess.py')
    if pre.exists():
        txt = pre.read_text(errors='ignore')
        target = 'trans_params = np.array([w0, h0, s, t[0], t[1]])'
        repl = """\n    try:\n        s = float(np.asarray(s).reshape(-1)[0])\n        t = np.asarray(t).reshape(-1)\n        tx = float(t[0]); ty = float(t[1])\n    except Exception:\n        tx, ty = t[0], t[1]\n    trans_params = np.array([float(w0), float(h0), s, tx, ty], dtype=np.float32)"""
        if target in txt: pre.write_text(txt.replace(target, repl))

manifest = {'engine':'SadTalker patched from existing audio','errors':{}}
try:
    sh(['apt-get','update','-qq'])
    sh(['apt-get','install','-y','-qq','ffmpeg','git','wget','libgl1','libglib2.0-0'])
    sh([sys.executable,'-m','pip','uninstall','-y','torch','torchaudio','torchvision','torchtext','torchdata'])
    sh([sys.executable,'-m','pip','install','-q','--no-cache-dir','--force-reinstall','torch==2.5.1+cu121','torchvision==0.20.1+cu121','torchaudio==2.5.1+cu121','--index-url','https://download.pytorch.org/whl/cu121'])
    sh([sys.executable,'-m','pip','install','-q','--no-cache-dir','--force-reinstall','numpy==1.26.4','pillow==11.3.0'])
    sh([sys.executable,'-m','pip','install','-q','--no-cache-dir','opencv-python','imageio','imageio-ffmpeg','scipy','librosa','pydub','tqdm','yacs','pyyaml','joblib','scikit-image','kornia','face-alignment','gfpgan','facexlib','basicsr','realesrgan','ffmpeg-python'])
    import torch
    manifest['cuda_available'] = bool(torch.cuda.is_available())
    manifest['cuda_device_name'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    face = find_asset('portrait.jpg')
    audio = find_asset('audio.wav')
    audio16 = WORK / 'audio_16k.wav'
    sh(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(audio),'-ar','16000','-ac','1','-af','loudnorm=I=-18:TP=-2:LRA=11',str(audio16)])
    os.chdir(WORK)
    sh(['git','clone','--depth','1','https://github.com/OpenTalker/SadTalker.git'])
    os.chdir(WORK/'SadTalker')
    sh(['bash','scripts/download_models.sh'])
    patch_sadtalker()
    result_dir = WORK / 'sadtalker_results'
    # Add '--enhancer','gfpgan' only for short clips with enough RAM.
    sh([sys.executable,'inference.py','--driven_audio',str(audio16),'--source_image',str(face),'--result_dir',str(result_dir),'--still','--preprocess','full','--size','256'])
    best = max(result_dir.rglob('*.mp4'), key=lambda p: p.stat().st_size)
    raw = WORK / 'raw.mp4'; shutil.copy(best, raw)
    sh(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(raw),'-c:v','libx264','-pix_fmt','yuv420p','-crf','23','-preset','veryfast','-c:a','aac','-b:a','128k','-movflags','+faststart',str(FINAL)])
    manifest.update({'output':FINAL.name,'duration_seconds':out(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nk=1:nw=1',str(FINAL)]).strip(),'bytes':FINAL.stat().st_size})
except Exception:
    manifest['errors']['pipeline'] = traceback.format_exc(); print(manifest['errors']['pipeline']); MANIFEST.write_text(json.dumps(manifest, indent=2)); raise
MANIFEST.write_text(json.dumps(manifest, indent=2)); print(MANIFEST.read_text())
